# PIE Colab Retraining Server
This notebook runs retraining on Colab GPU and exposes HTTP endpoints with ngrok.

In [ ]:
!pip -q install flask pyngrok lightgbm optuna shap imbalanced-learn pydrive2 gdown scikit-learn pandas numpy

: 

In [ ]:
import json
import os
import pickle
import threading
import time
from datetime import datetime

import numpy as np
import optuna
import pandas as pd
import shap
from flask import Flask, Response, jsonify, request
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from pyngrok import ngrok
from scipy.stats import ks_2samp
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

In [ ]:
# Configure these paths to your Drive mount locations.
DRIVE_DATASET_PATH = '/content/drive/MyDrive/pie/training/latest_training_data.csv'
DRIVE_MODELS_DIR = '/content/drive/MyDrive/pie/models'
DRIVE_META_DIR = '/content/drive/MyDrive/pie/meta'
CURRENT_PROD_AUC = 0.0
DEFAULT_BANK_THRESHOLD = 0.3

os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
os.makedirs(DRIVE_META_DIR, exist_ok=True)

In [ ]:
app = Flask(__name__)
jobs = {}

def _append_log(job_id: str, message: str):
    jobs[job_id]['logs'].append(f"[{datetime.utcnow().isoformat()}] {message}")

def _load_training_data(path: str) -> pd.DataFrame:
    if path.endswith('.parquet'):
        return pd.read_parquet(path)
    return pd.read_csv(path)

def _evaluate(y_true, y_prob, threshold: float, bank_threshold: float):
    y_pred = (y_prob >= threshold).astype(int)
    y_pred_bank = (y_prob >= bank_threshold).astype(int)
    auc = float(roc_auc_score(y_true, y_prob))
    ks = float(ks_2samp(y_prob[y_true == 0], y_prob[y_true == 1]).statistic)
    precision = float(precision_score(y_true, y_pred, zero_division=0))
    recall = float(recall_score(y_true, y_pred, zero_division=0))
    f1 = float(f1_score(y_true, y_pred, zero_division=0))
    precision_bank = float(precision_score(y_true, y_pred_bank, zero_division=0))
    recall_bank = float(recall_score(y_true, y_pred_bank, zero_division=0))
    gini = float(2 * auc - 1)
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {
        'auc_roc': auc,
        'gini': gini,
        'ks_stat': ks,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_bank': precision_bank,
        'recall_bank': recall_bank,
        'confusion_matrix': cm,
    }

def _best_threshold(y_true, y_prob):
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.1, 0.9, 41):
        score = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if score > best_f1:
            best_t, best_f1 = float(t), float(score)
    return best_t

In [ ]:
def _train_job(job_id: str, payload: dict):
    try:
        jobs[job_id]['status'] = 'running'
        bank_threshold = float(payload.get('bank_threshold', DEFAULT_BANK_THRESHOLD))
        current_prod_auc = float(payload.get('current_prod_auc', CURRENT_PROD_AUC))

        _append_log(job_id, 'Loading dataset from Drive...')
        df = _load_training_data(payload.get('dataset_path', DRIVE_DATASET_PATH))
        target_col = payload.get('target_column', 'target')
        if target_col not in df.columns:
            raise ValueError(f'Missing target column: {target_col}')

        X = df.drop(columns=[target_col]).replace([np.inf, -np.inf], np.nan).fillna(0)
        y = df[target_col].astype(int)
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        _append_log(job_id, 'Applying SMOTE on training split...')
        X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

        _append_log(job_id, 'Starting Optuna tuning (30 trials)...')
        trial_count = int(payload.get('optuna_trials', 30))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 150, 500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 20, 120),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'random_state': 42,
                'n_jobs': -1,
            }
            model = LGBMClassifier(**params)
            fold_aucs = []
            for train_idx, valid_idx in cv.split(X_res, y_res):
                X_tr, X_va = X_res.iloc[train_idx], X_res.iloc[valid_idx]
                y_tr, y_va = y_res.iloc[train_idx], y_res.iloc[valid_idx]
                model.fit(X_tr, y_tr)
                prob = model.predict_proba(X_va)[:, 1]
                fold_aucs.append(roc_auc_score(y_va, prob))
            jobs[job_id]['progress'] = int((trial.number + 1) / trial_count * 100)
            jobs[job_id]['trial'] = trial.number + 1
            return float(np.mean(fold_aucs))

        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=trial_count)

        _append_log(job_id, f"Best AUC from tuning: {study.best_value:.4f}")
        best_model = LGBMClassifier(**study.best_params, random_state=42, n_jobs=-1)
        best_model.fit(X_res, y_res)

        y_prob = best_model.predict_proba(X_val)[:, 1]
        best_threshold = _best_threshold(y_val, y_prob)
        metrics = _evaluate(y_val, y_prob, best_threshold, bank_threshold)

        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_val.iloc[:300])
        shap_arr = shap_values[1] if isinstance(shap_values, list) else shap_values
        shap_importance = np.abs(shap_arr).mean(axis=0)
        top_idx = np.argsort(shap_importance)[::-1][:15]
        top_features = [{
            'feature': X.columns[i],
            'importance': float(shap_importance[i])
        } for i in top_idx]

        timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
        version = payload.get('version') or f'v{timestamp}'
        model_name = f'lightgbm_pie_{version}.pkl'
        model_path = os.path.join(DRIVE_MODELS_DIR, model_name)
        threshold_path = os.path.join(DRIVE_MODELS_DIR, f'lightgbm_pie_{version}_threshold.pkl')
        features_path = os.path.join(DRIVE_MODELS_DIR, f'lightgbm_pie_{version}_features.pkl')

        if metrics['auc_roc'] <= current_prod_auc:
            jobs[job_id]['status'] = 'failed'
            jobs[job_id]['result'] = {
                'message': 'Candidate underperforms current production model',
                'metrics': metrics,
                'current_production_auc': current_prod_auc,
            }
            _append_log(job_id, 'Candidate underperformed. Keeping production model.')
            return

        with open(model_path, 'wb') as f:
            pickle.dump(best_model, f)
        with open(threshold_path, 'wb') as f:
            pickle.dump(best_threshold, f)
        with open(features_path, 'wb') as f:
            pickle.dump(list(X.columns), f)

        meta_path = os.path.join(DRIVE_META_DIR, f'lightgbm_pie_{version}_meta.json')
        with open(meta_path, 'w') as f:
            json.dump({
                'version': version,
                'trained_at': datetime.utcnow().isoformat(),
                'metrics': metrics,
                'threshold': best_threshold,
                'top_shap_features': top_features,
                'optuna_best_value': float(study.best_value),
                'optuna_best_params': study.best_params,
            }, f, indent=2)

        jobs[job_id]['status'] = 'done'
        jobs[job_id]['result'] = {
            'message': 'Retraining successful',
            'version': version,
            'metrics': metrics,
            'drive_file_id': payload.get('drive_file_id', model_name),
            'features_file_id': payload.get('features_file_id', f'lightgbm_pie_{version}_features.pkl'),
            'threshold_file_id': payload.get('threshold_file_id', f'lightgbm_pie_{version}_threshold.pkl'),
            'meta_path': meta_path,
        }
        _append_log(job_id, 'Training completed successfully.')
    except Exception as exc:
        jobs[job_id]['status'] = 'failed'
        jobs[job_id]['result'] = {'error': str(exc)}
        _append_log(job_id, f'Error: {exc}')

In [ ]:
@app.get('/health')
def health():
    return {'status': 'ok', 'service': 'pie-colab-retrain'}

@app.post('/retrain')
def retrain():
    payload = request.get_json(force=True, silent=True) or {}
    job_id = str(payload.get('job_id') or datetime.utcnow().strftime('%Y%m%d%H%M%S'))
    jobs[job_id] = {'status': 'pending', 'logs': [], 'progress': 0, 'trial': 0, 'result': None}
    t = threading.Thread(target=_train_job, args=(job_id, payload), daemon=True)
    t.start()
    return jsonify({'job_id': job_id, 'status': 'running'})

@app.get('/status/<job_id>')
def status(job_id):
    if job_id not in jobs:
        return jsonify({'status': 'missing'}), 404
    job = jobs[job_id]
    return jsonify({
        'job_id': job_id,
        'status': job['status'],
        'progress': job.get('progress', 0),
        'trial': job.get('trial', 0),
        'best_auc_so_far': job.get('result', {}).get('metrics', {}).get('auc_roc') if job.get('result') else None,
        'result': job.get('result'),
    })

@app.get('/logs/<job_id>')
def logs(job_id):
    if job_id not in jobs:
        return jsonify({'status': 'missing'}), 404

    def stream():
        cursor = 0
        while True:
            data = jobs.get(job_id)
            if not data:
                break
            lines = data.get('logs', [])
            while cursor < len(lines):
                yield f"data: {lines[cursor]}\n\n"
                cursor += 1
            if data.get('status') in {'done', 'failed'}:
                yield f"data: [job-{data['status']}]\n\n"
                break
            time.sleep(1)

    return Response(stream(), mimetype='text/event-stream')

In [ ]:
public_url = ngrok.connect(addr=5000, bind_tls=True).public_url
print('Colab ngrok URL:', public_url)
app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)